# Biomedical waste: model comparison and prediction

This notebook reads the real `State-wise 6 Years` sheet in the supplied Excel workbook. It trains candidate models on 2018-2020 and tests them on 2021-2023. It selects the model with the lowest RMSE on that holdout, refits it on all observed years through 2023, and estimates state-wise BMW generation for 2024-2027.

**Interpretation and limitations:** predictions use reporting year and state/UT. Government-hospital beds are excluded because they do not represent all hospital beds, and future bed counts are unavailable. Treatment/gap fields are excluded to avoid target leakage. Models are trained on 2018-2020 and tested on 2021-2023; the lowest-error model is then refit on all observed years for 2024-2027 estimates. Validation cannot guarantee future accuracy, especially where source values change abruptly or disagree. Tree models do not extrapolate trends naturally. Missing target values remain missing and are excluded from model fitting, never changed to zero. Pandas is intentionally not used because Windows Application Control blocks its native extension in this environment.

In [38]:
from pathlib import Path
import csv
import math
from html import escape

import numpy as np
import joblib
from openpyxl import load_workbook
from IPython.display import HTML, display
from sklearn.ensemble import (
    AdaBoostRegressor,
    BaggingRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import (
    GammaRegressor, HuberRegressor, Lasso, LinearRegression,
    PoissonRegressor, Ridge, TweedieRegressor,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

WORKBOOK_NAME = "biomedical dataset.xlsx"
SHEET_NAME = "State-wise 6 Years"
TARGET_COLUMN = "BMW_Generated_kg_per_day_as_reported"
TREATMENT_COLUMN = "BMW_Treated_kg_per_day_as_reported"
YEAR_COLUMN = "Year"
STATE_COLUMN = "State_UT_or_Reporting_Entity"
MISSING_MARKERS = {"", "na", "n/a", "not available", "not reported"}

project_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "datasets" / WORKBOOK_NAME).is_file()
     or (parent / "dataset" / WORKBOOK_NAME).is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        f"Could not locate {WORKBOOK_NAME!r} in a dataset/ or datasets/ folder "
        "from this notebook's working directory."
    )

workbook_path = next(
    path for path in (project_root / "datasets" / WORKBOOK_NAME,
                      project_root / "dataset" / WORKBOOK_NAME)
    if path.is_file()
)
workbook = load_workbook(workbook_path, read_only=True, data_only=True)
try:
    if SHEET_NAME not in workbook.sheetnames:
        raise ValueError(f"Required sheet {SHEET_NAME!r} not found. Available sheets: {workbook.sheetnames}")
    sheet = workbook[SHEET_NAME]
    rows = sheet.iter_rows(values_only=True)
    headers = [str(value).strip() if value is not None else "" for value in next(rows)]
    required = {YEAR_COLUMN, STATE_COLUMN, TARGET_COLUMN, TREATMENT_COLUMN}
    missing_columns = required.difference(headers)
    if missing_columns:
        raise ValueError(f"Required columns missing from {SHEET_NAME!r}: {sorted(missing_columns)}")

    records = []
    for row in rows:
        if not any(value is not None for value in row):
            continue
        record = dict(zip(headers, row))
        state = record.get(STATE_COLUMN)
        year = record.get(YEAR_COLUMN)
        target = record.get(TARGET_COLUMN)
        treated = record.get(TREATMENT_COLUMN)
        if not isinstance(state, str) or not state.strip():
            continue
        try:
            year = int(year)
        except (TypeError, ValueError):
            continue
        if target is None or (isinstance(target, str) and target.strip().casefold() in MISSING_MARKERS):
            target = None
        else:
            try:
                target = float(target)
            except (TypeError, ValueError):
                target = None
            if target is not None and (not math.isfinite(target) or target < 0):
                target = None
        if treated is None or (isinstance(treated, str) and treated.strip().casefold() in MISSING_MARKERS):
            treated = None
        else:
            try:
                treated = float(treated)
            except (TypeError, ValueError):
                treated = None
            if treated is not None and (not math.isfinite(treated) or treated < 0):
                treated = None
        records.append({"Year": year, "State": state.strip(), "Waste_kg_day": target,
                        "Treated_kg_day": treated})
finally:
    workbook.close()

observed = [row for row in records if row["Waste_kg_day"] is not None]
if not observed:
    raise ValueError("No valid numeric waste-generation observations were found.")
years = sorted({row["Year"] for row in observed})
training_years = [2018, 2019, 2020]
test_years = [2021, 2022, 2023]
if not set(training_years + test_years).issubset(years):
    raise ValueError("The dataset must contain observations for 2018-2023.")
training_rows = [row for row in observed if row["Year"] in training_years]
test_rows = [row for row in observed if row["Year"] in test_years]
if not training_rows or not test_rows:
    raise ValueError("Chronological split produced an empty training or test set.")

print(f"Workbook: {workbook_path.relative_to(project_root)}")
print(f"Sheet: {SHEET_NAME}; panel rows: {len(records)}; numeric target rows: {len(observed)}")
print(f"Validation: train on {training_years[0]}-{training_years[-1]}; test on {test_years[0]}-{test_years[-1]} "
      f"({len(training_rows)} training rows, {len(test_rows)} test rows).")
print("Predictors: Year and State/UT only. Missing target values were excluded, not imputed.")

Workbook: datasets\biomedical dataset.xlsx
Sheet: State-wise 6 Years; panel rows: 216; numeric target rows: 210
Validation: train on 2018-2020; test on 2021-2023 (104 training rows, 106 test rows).
Predictors: Year and State/UT only. Missing target values were excluded, not imputed.


In [39]:
class LastObservationBaseline:
    def fit(self, X, y):
        self.latest_by_state = {}
        self.latest_year_by_state = {}
        self.default_value = float(np.mean(y))
        for features, target in zip(X, y):
            active = np.flatnonzero(features[1:])
            if len(active) == 0:
                continue
            state_index = int(active[0])
            if features[0] >= self.latest_year_by_state.get(state_index, -np.inf):
                self.latest_year_by_state[state_index] = features[0]
                self.latest_by_state[state_index] = float(target)
        return self

    def predict(self, X):
        predictions = []
        for features in X:
            active = np.flatnonzero(features[1:])
            state_index = int(active[0]) if len(active) else None
            predictions.append(self.latest_by_state.get(state_index, self.default_value))
        return np.asarray(predictions, dtype=float)

model_factories = {
    "Linear Regression": lambda: LinearRegression(),
    "Log-linear Regression": lambda: TransformedTargetRegressor(
        regressor=LinearRegression(), func=np.log1p, inverse_func=np.expm1
    ),
    "Poisson GLM": lambda: PoissonRegressor(alpha=1, max_iter=1000),
    "Decision Tree": lambda: DecisionTreeRegressor(
        max_depth=4, min_samples_leaf=4, random_state=42
    ),
    "Random Forest": lambda: RandomForestRegressor(
        n_estimators=250, min_samples_leaf=2, random_state=42, n_jobs=-1
    ),
    "Random Forest (leaf=1)": lambda: RandomForestRegressor(
        n_estimators=300, min_samples_leaf=1, random_state=42, n_jobs=-1
    ),
    "Extra Trees": lambda: ExtraTreesRegressor(
        n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=-1
    ),
    "Extra Trees (leaf=1)": lambda: ExtraTreesRegressor(
        n_estimators=300, min_samples_leaf=1, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": lambda: GradientBoostingRegressor(
        n_estimators=100, learning_rate=0.05, max_depth=2,
        min_samples_leaf=3, random_state=42
    ),
    "Hist. Gradient Boosting": lambda: HistGradientBoostingRegressor(
        max_iter=150, max_leaf_nodes=7, l2_regularization=1.0, random_state=42
    ),
    "KNN (k=5, distance)": lambda: make_pipeline(
        StandardScaler(), KNeighborsRegressor(n_neighbors=5, weights="distance")
    ),
    "Lasso": lambda: Lasso(alpha=100, max_iter=10000),
    "Ridge": lambda: Ridge(alpha=1),
    "AdaBoost": lambda: AdaBoostRegressor(
        n_estimators=100, learning_rate=0.05, random_state=42
    ),
    "Bagging (trees)": lambda: BaggingRegressor(
        estimator=DecisionTreeRegressor(max_depth=5, min_samples_leaf=2),
        n_estimators=100, random_state=42, n_jobs=-1
    ),
    "SVR (RBF)": lambda: make_pipeline(
        StandardScaler(), SVR(C=100, epsilon=0.1, kernel="rbf")
    ),
    "Log Ridge": lambda: TransformedTargetRegressor(
        regressor=Ridge(alpha=1), func=np.log1p, inverse_func=np.expm1
    ),
    "Log Random Forest": lambda: TransformedTargetRegressor(
        regressor=RandomForestRegressor(
            n_estimators=250, min_samples_leaf=2, random_state=42, n_jobs=-1
        ), func=np.log1p, inverse_func=np.expm1
    ),
    "Log Extra Trees": lambda: TransformedTargetRegressor(
        regressor=ExtraTreesRegressor(
            n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=-1
        ), func=np.log1p, inverse_func=np.expm1
    ),
    "Log Extra Trees leaf=1": lambda: TransformedTargetRegressor(
        regressor=ExtraTreesRegressor(
            n_estimators=300, min_samples_leaf=1, random_state=42, n_jobs=-1
        ), func=np.log1p, inverse_func=np.expm1
    ),
    "Log Gradient Boosting": lambda: TransformedTargetRegressor(
        regressor=GradientBoostingRegressor(
            n_estimators=100, learning_rate=0.05, max_depth=2,
            min_samples_leaf=3, random_state=42
        ), func=np.log1p, inverse_func=np.expm1
    ),
    "Log Hist Gradient Boosting": lambda: TransformedTargetRegressor(
        regressor=HistGradientBoostingRegressor(
            max_iter=150, max_leaf_nodes=7, l2_regularization=1.0, random_state=42
        ), func=np.log1p, inverse_func=np.expm1
    ),
    "Log KNN": lambda: TransformedTargetRegressor(
        regressor=make_pipeline(
            StandardScaler(), KNeighborsRegressor(n_neighbors=5, weights="distance")
        ), func=np.log1p, inverse_func=np.expm1
    ),
    "Log SVR": lambda: TransformedTargetRegressor(
        regressor=make_pipeline(
            StandardScaler(), SVR(C=100, epsilon=0.1, kernel="rbf")
        ), func=np.log1p, inverse_func=np.expm1
    ),
    "Huber log": lambda: TransformedTargetRegressor(
        regressor=HuberRegressor(epsilon=1.35, alpha=0.0001, max_iter=1000),
        func=np.log1p, inverse_func=np.expm1
    ),
    "Gamma GLM": lambda: GammaRegressor(alpha=1, max_iter=1000),
    "Tweedie GLM": lambda: TweedieRegressor(
        power=1.5, alpha=1, link="log", max_iter=1000
    ),
    "Last-observation baseline": lambda: LastObservationBaseline(),
}

def build_model(regressor):
    return regressor

def encode_features(rows, state_levels, base_year):
    """Encode each row as numeric year offset plus one-hot state features."""
    state_index = {state: index for index, state in enumerate(state_levels)}
    features = np.zeros((len(rows), len(state_levels) + 1), dtype=float)
    for row_index, row in enumerate(rows):
        features[row_index, 0] = row["Year"] - base_year
        state_index_for_row = state_index.get(row["State"])
        if state_index_for_row is not None:
            features[row_index, state_index_for_row + 1] = 1.0
    return features

train_states = sorted({row["State"] for row in training_rows})
base_year = min(row["Year"] for row in training_rows)
X_train = encode_features(training_rows, train_states, base_year)
y_train = np.asarray([row["Waste_kg_day"] for row in training_rows], dtype=float)
X_test = encode_features(test_rows, train_states, base_year)
y_test = np.asarray([row["Waste_kg_day"] for row in test_rows], dtype=float)

validation_predictions = {}
comparison_rows = []
for model_name, factory in model_factories.items():
    model = build_model(factory())
    model.fit(X_train, y_train)
    predictions = np.maximum(0.0, model.predict(X_test))
    validation_predictions[model_name] = predictions
    comparison_rows.append({
        "Model": model_name,
        "MAE (kg/day)": mean_absolute_error(y_test, predictions),
        "RMSE (kg/day)": math.sqrt(mean_squared_error(y_test, predictions)),
        "R²": r2_score(y_test, predictions) if len(y_test) > 1 else float("nan"),
    })

comparison_rows.sort(key=lambda row: row["RMSE (kg/day)"])
best_model_name = comparison_rows[0]["Model"]

def display_table(title, columns, rows, formats=None):
    formats = formats or {}
    header = "".join(f"<th>{escape(str(column))}</th>" for column in columns)
    body_rows = []
    for row in rows:
        cells = []
        for column in columns:
            value = row[column]
            if isinstance(value, (int, float, np.number)) and math.isfinite(float(value)):
                if column in {"Year", "Last reported year", "Forecast year", "Test year", "Reporting entities", "Numeric generation values used", "Missing generation values"}:
                    value = str(int(value))
                elif column == "Change vs last actual (%)":
                    value = f"{float(value):+,.1f}%"
                else:
                    value = format(float(value), formats.get(column, ",.2f"))
            else:
                value = str(value)
            cells.append(f"<td>{escape(value)}</td>")
        body_rows.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML(
        f"<h3>{escape(title)}</h3><table style='border-collapse:collapse' "
        "border='1' cellpadding='6'><thead><tr>" + header + "</tr></thead><tbody>"
        + "".join(body_rows) + "</tbody></table>"
    ))

display_table(
    f"Chronological model comparison (test years {test_years[0]}-{test_years[-1]}; lower RMSE selects the forecast model)",
    ["Model", "MAE (kg/day)", "RMSE (kg/day)", "R²"],
    comparison_rows,
    {"MAE (kg/day)": ",.1f", "RMSE (kg/day)": ",.1f", "R²": ".3f"},
)

# Compare models across the latest three available chronological holdout years.
recent_holdout_years = years[-3:]
rolling_scores = []
for validation_year in recent_holdout_years:
    fold_train = [row for row in observed if row["Year"] < validation_year]
    if not fold_train:
        continue
    fold_states = sorted({row["State"] for row in fold_train})
    known_states = set(fold_states)
    fold_test = [
        row for row in observed
        if row["Year"] == validation_year and row["State"] in known_states
    ]
    if not fold_test:
        continue
    fold_base_year = min(row["Year"] for row in fold_train)
    X_fold_train = encode_features(fold_train, fold_states, fold_base_year)
    X_fold_test = encode_features(fold_test, fold_states, fold_base_year)
    y_fold_train = np.asarray([row["Waste_kg_day"] for row in fold_train], dtype=float)
    y_fold_test = np.asarray([row["Waste_kg_day"] for row in fold_test], dtype=float)
    for model_name, factory in model_factories.items():
        fold_model = build_model(factory())
        fold_model.fit(X_fold_train, y_fold_train)
        fold_prediction = np.maximum(0.0, fold_model.predict(X_fold_test))
        rolling_scores.append({
            "Test year": validation_year,
            "Model": model_name,
            "R²": r2_score(y_fold_test, fold_prediction)
            if len(y_fold_test) > 1 else float("nan"),
            "MAE (kg/day)": mean_absolute_error(y_fold_test, fold_prediction),
            "RMSE (kg/day)": math.sqrt(mean_squared_error(y_fold_test, fold_prediction)),
        })

rolling_summary = []
for model_name in model_factories:
    scores = [row for row in rolling_scores if row["Model"] == model_name]
    summary = {
        "Model": model_name,
        "Mean R² (recent years)": float(np.mean([row["R²"] for row in scores])),
        "Mean MAE (kg/day)": float(np.mean([row["MAE (kg/day)" ] for row in scores])),
        "Mean RMSE (kg/day)": float(np.mean([row["RMSE (kg/day)" ] for row in scores])),
    }
    summary.update({
        f"R² {row['Test year']}": row["R²"] for row in scores
    })
    rolling_summary.append(summary)
rolling_summary.sort(key=lambda row: row["Mean RMSE (kg/day)"])
rolling_best_model_name = rolling_summary[0]["Model"]
rolling_columns = [
    "Model", *[f"R² {year}" for year in recent_holdout_years],
    "Mean R² (recent years)", "Mean MAE (kg/day)", "Mean RMSE (kg/day)",
]
display_table(
    "Supplemental rolling comparison (the fixed 2021-2023 holdout selects the forecast model)",
rolling_columns,
rolling_summary,
 {"Mean R² (recent years)": ".3f", "Mean MAE (kg/day)": ",.1f",
  "Mean RMSE (kg/day)": ",.1f", **{f"R² {year}": ".3f" for year in recent_holdout_years}},
)
print(
    f"Selected for future estimates: {best_model_name}, based on lowest RMSE "
    f"on the fixed {test_years[0]}-{test_years[-1]} test set. "
    f"The supplemental rolling comparison's lowest-mean-RMSE model was {rolling_best_model_name}."
)

Model,MAE (kg/day),RMSE (kg/day),R²
Poisson GLM,"3,629.0","8,268.4",0.885
Linear Regression,"4,232.7","8,558.2",0.876
Log-linear Regression,"3,932.7","8,601.9",0.875
Extra Trees,"3,925.9","8,661.6",0.873
Huber log,"4,157.4","8,756.6",0.871
Log Extra Trees,"4,438.4","8,887.5",0.867
Random Forest (leaf=1),"4,299.3","9,445.2",0.849
Extra Trees (leaf=1),"4,164.6","9,710.0",0.841
Last-observation baseline,"4,165.4","9,710.0",0.841
Log Extra Trees leaf=1,"4,165.4","9,710.0",0.841


Model,R² 2021,R² 2022,R² 2023,Mean R² (recent years),Mean MAE (kg/day),Mean RMSE (kg/day)
Log Extra Trees leaf=1,0.963,0.838,0.985,0.929,"2,465.5","5,739.8"
Extra Trees (leaf=1),0.963,0.837,0.985,0.929,"2,470.9","5,745.7"
Last-observation baseline,0.963,0.837,0.985,0.929,"2,472.4","5,745.8"
Random Forest (leaf=1),0.950,0.843,0.956,0.916,"3,033.0","6,694.2"
Log-linear Regression,0.969,0.861,0.905,0.912,"3,200.0","6,950.9"
Log Extra Trees,0.948,0.855,0.922,0.908,"3,595.0","7,188.3"
Poisson GLM,0.968,0.857,0.892,0.906,"3,236.2","7,197.5"
Extra Trees,0.955,0.855,0.905,0.905,"3,251.0","7,307.9"
Linear Regression,0.959,0.857,0.884,0.900,"3,692.9","7,477.6"
"KNN (k=5, distance)",0.921,0.830,0.928,0.893,"4,152.4","7,756.4"


Selected for future estimates: Poisson GLM, based on lowest RMSE on the fixed 2021-2023 test set. The supplemental rolling comparison's lowest-mean-RMSE model was Log Extra Trees leaf=1.


In [40]:
yearly_summary = []
all_years = sorted({row["Year"] for row in records})
for year in all_years:
    reporting_rows = [row for row in records if row["Year"] == year]
    year_rows = [row for row in observed if row["Year"] == year]
    reporting_entities = len({row["State"] for row in reporting_rows})
    yearly_summary.append({
        "Year": year,
        "Reporting entities": reporting_entities,
        "Numeric generation values used": len(year_rows),
        "Missing generation values": reporting_entities - len(year_rows),
        "Generated total (kg/day)": sum(row["Waste_kg_day"] for row in year_rows),
        "Mean generated (kg/day)": float(np.mean([row["Waste_kg_day"] for row in year_rows])) if year_rows else None,
    })
display_table(
    "Historical generation and reporting coverage by year",
    ["Year", "Reporting entities", "Numeric generation values used", "Missing generation values", "Generated total (kg/day)", "Mean generated (kg/day)"],
    yearly_summary,
    {"Generated total (kg/day)": ",.1f", "Mean generated (kg/day)": ",.1f"},
)

state_summary = []
for state in sorted({row["State"] for row in observed}):
    state_rows = [row for row in observed if row["State"] == state]
    state_summary.append({
        "State/UT": state,
        "Observed years": len(state_rows),
        "Mean generated (kg/day)": float(np.mean([row["Waste_kg_day"] for row in state_rows])),
    })
state_summary.sort(key=lambda row: row["Mean generated (kg/day)"], reverse=True)
display_table(
    "Top 10 reporting entities by historical average generation",
    ["State/UT", "Observed years", "Mean generated (kg/day)"],
    state_summary[:10],
    {"Mean generated (kg/day)": ",.1f"},
)

actual_prediction_rows = []
holdout_predictions = validation_predictions[best_model_name]
for row, prediction in zip(test_rows, holdout_predictions):
    actual_prediction_rows.append({
        "Year": row["Year"],
        "State/UT": row["State"],
        "Actual generated (kg/day)": row["Waste_kg_day"],
        "Predicted generated (kg/day)": float(prediction),
        "Prediction error (kg/day)": row["Waste_kg_day"] - float(prediction),
    })
display_table(
    f"Actual vs predicted values for {test_years[0]}-{test_years[-1]} using {best_model_name}",
    ["Year", "State/UT", "Actual generated (kg/day)",
     "Predicted generated (kg/day)", "Prediction error (kg/day)"],
    actual_prediction_rows,
    {"Actual generated (kg/day)": ",.1f",
     "Predicted generated (kg/day)": ",.1f",
     "Prediction error (kg/day)": ",.1f"},
)


Year,Reporting entities,Numeric generation values used,Missing generation values,Generated total (kg/day),Mean generated (kg/day)
2018,36,35,1,"608,392.6","17,382.6"
2019,36,35,1,"613,370.7","17,524.9"
2020,36,34,2,"607,260.1","17,860.6"
2021,36,34,2,"657,933.0","19,351.0"
2022,36,36,0,"651,725.1","18,103.5"
2023,36,36,0,"686,381.0","19,066.1"


State/UT,Observed years,Mean generated (kg/day)
Karnataka,6.00,"75,977.4"
Maharashtra,6.00,"73,207.2"
Uttar Pradesh,6.00,"71,438.0"
Kerala,6.00,"57,782.9"
Tamil Nadu,6.00,"49,156.2"
West Bengal,5.00,"39,997.8"
Gujarat,6.00,"28,850.2"
Bihar,6.00,"28,721.4"
Delhi,6.00,"27,569.6"
Telangana,6.00,"22,730.3"


Year,State/UT,Actual generated (kg/day),Predicted generated (kg/day),Prediction error (kg/day)
2021,Andaman and Nicobar Islands,543.0,593.1,-50.1
2021,Assam,"8,421.0","8,808.7",-387.7
2021,Bihar,"20,549.0","34,493.1","-13,944.1"
2021,Chandigarh,"5,374.0","4,547.1",826.9
2021,Chhattisgarh,"7,907.0","10,760.2","-2,853.2"
2021,Daman and Diu and Dadra and Nagar Haveli,554.0,488.2,65.8
2021,Delhi,"25,828.0","27,861.5","-2,033.5"
2021,Goa,"1,129.0","1,681.4",-552.4
2021,Gujarat,"52,800.0","42,336.5","10,463.5"
2021,Haryana,"21,436.0","17,068.3","4,367.7"


## Multi-year estimates

The notebook produces state-wise estimates for 2024, 2025, 2026, and 2027. Candidate models are compared using the fixed 2018-2020 training and 2021-2023 test split; the lowest-RMSE model is then refit on all observed 2018-2023 numeric targets. A trend-capable model such as Poisson GLM can produce year-varying estimates beyond the historical period, unlike tree models which generally cannot extrapolate. Forecasts become more uncertain farther into the future. The panel is small and partly unaudited, and no future-year features beyond year/state are available, so treat estimates as exploratory rather than measured data.

In [41]:
forecast_years = [2024, 2025, 2026, 2027]
all_states = sorted({row["State"] for row in observed})
last_observation_by_state = {}
for row in observed:
    previous = last_observation_by_state.get(row["State"])
    if previous is None or row["Year"] > previous["Year"]:
        last_observation_by_state[row["State"]] = row
forecast_rows_by_model = {}
final_table_rows = []

for model_name, factory in model_factories.items():
    final_model = build_model(factory())
    final_states = sorted({row["State"] for row in observed})
    final_base_year = min(row["Year"] for row in observed)
    final_model.fit(
        encode_features(observed, final_states, final_base_year),
        np.asarray([row["Waste_kg_day"] for row in observed], dtype=float),
    )
    model_forecasts = {}
    for forecast_year in forecast_years:
        final_features = encode_features(
            [{"Year": forecast_year, "State": state} for state in all_states],
            final_states,
            final_base_year,
        )
        model_forecasts[forecast_year] = np.maximum(
            0.0, final_model.predict(final_features)
        )
    forecast_rows_by_model[model_name] = model_forecasts
    metric_row = next(row for row in comparison_rows if row["Model"] == model_name)
    final_table_row = {
        "Model": model_name,
        "Validation MAE (kg/day)": metric_row["MAE (kg/day)"],
        "Validation RMSE (kg/day)": metric_row["RMSE (kg/day)"],
        "Lowest RMSE (2021-2023 holdout)": "Yes" if model_name == best_model_name else "No",
    }
    final_table_row.update({
        f"{forecast_year} total (kg/day)": float(np.sum(model_forecasts[forecast_year]))
        for forecast_year in forecast_years
    })
    final_table_rows.append(final_table_row)

forecast_total_columns = [f"{year} total (kg/day)" for year in forecast_years]
display_table(
    "Model metrics and estimated state totals for 2024–2027",
    ["Model", "Validation MAE (kg/day)", "Validation RMSE (kg/day)",
     *forecast_total_columns, "Lowest RMSE (2021-2023 holdout)"],
    final_table_rows,
    {"Validation MAE (kg/day)": ",.1f", "Validation RMSE (kg/day)": ",.1f",
     **{column: ",.1f" for column in forecast_total_columns}},
)

forecast_model_name = best_model_name
selected_forecasts = forecast_rows_by_model[forecast_model_name]
latest_actual_values = np.asarray(
    [last_observation_by_state[state]["Waste_kg_day"] for state in all_states],
    dtype=float,
)
print(
    f"State-wise future estimates use {forecast_model_name}, selected by lowest "
    f"RMSE on the {test_years[0]}-{test_years[-1]} holdout and refit on 2018-2023 data."
)
state_forecast_rows = []
for forecast_year in forecast_years:
    for state, value in zip(all_states, selected_forecasts[forecast_year]):
        last_actual = last_observation_by_state[state]
        previous_value = last_actual["Waste_kg_day"]
        change_percent = (
            (float(value) - previous_value) / previous_value * 100.0
            if previous_value != 0
            else None
        )
        state_forecast_rows.append({
            "State/UT": state,
            "Last reported year": last_actual["Year"],
            "Last actual generation (kg/day)": previous_value,
            "Forecast year": forecast_year,
            "Predicted generation (kg/day)": float(value),
            "Change vs last actual (%)": change_percent,
            "Review note": (
                "Large difference; verify source data"
                if change_percent is not None and abs(change_percent) > 50
                else ""
            ),
            "Model": forecast_model_name,
        })
state_forecast_rows.sort(
    key=lambda row: abs(row["Change vs last actual (%)"])
    if row["Change vs last actual (%)"] is not None else -1,
    reverse=True,
)
display_table(
    f"State-wise historical-trend estimates for 2024–2027 ({forecast_model_name})",
    ["State/UT", "Last reported year", "Last actual generation (kg/day)",
     "Forecast year", "Predicted generation (kg/day)",
     "Change vs last actual (%)", "Review note", "Model"],
    state_forecast_rows,
    {"Last actual generation (kg/day)": ",.1f",
     "Predicted generation (kg/day)": ",.1f",
     "Change vs last actual (%)": "+,.1f"},
)
print("Forecast values are model outputs, not observed measurements. Treat them as exploratory only; "
      "future hospital capacity, population, and policy changes are unavailable.")

Model,Validation MAE (kg/day),Validation RMSE (kg/day),2024 total (kg/day),2025 total (kg/day),2026 total (kg/day),2027 total (kg/day),Lowest RMSE (2021-2023 holdout)
Linear Regression,"4,232.7","8,558.2","701,390.0","716,962.2","732,534.5","748,106.7",No
Log-linear Regression,"3,932.7","8,601.9","592,625.7","585,523.8","578,507.0","571,574.3",No
Poisson GLM,"3,629.0","8,268.4","702,312.6","719,029.3","736,144.0","753,666.1",Yes
Decision Tree,"18,051.5","24,355.8","649,094.4","649,094.4","649,094.4","649,094.4",No
Random Forest,"8,775.6","13,117.2","647,341.2","647,341.2","647,341.2","647,341.2",No
Random Forest (leaf=1),"4,299.3","9,445.2","673,866.2","673,866.2","673,866.2","673,866.2",No
Extra Trees,"3,925.9","8,661.6","660,224.4","660,224.4","660,224.4","660,224.4",No
Extra Trees (leaf=1),"4,164.6","9,710.0","686,381.0","686,381.0","686,381.0","686,381.0",No
Gradient Boosting,"11,055.2","13,899.7","665,739.6","665,739.6","665,739.6","665,739.6",No
Hist. Gradient Boosting,"18,051.5","24,355.8","686,381.0","686,381.0","686,381.0","686,381.0",No


State-wise future estimates use Poisson GLM, selected by lowest RMSE on the 2021-2023 holdout and refit on 2018-2023 data.


State/UT,Last reported year,Last actual generation (kg/day),Forecast year,Predicted generation (kg/day),Change vs last actual (%),Review note,Model
Gujarat,2023,358.0,2027,"33,528.3","+9,264.7%",Large difference; verify source data,Poisson GLM
Gujarat,2023,358.0,2026,"32,748.8","+9,046.9%",Large difference; verify source data,Poisson GLM
Gujarat,2023,358.0,2025,"31,987.4","+8,834.3%",Large difference; verify source data,Poisson GLM
Gujarat,2023,358.0,2024,"31,243.7","+8,626.6%",Large difference; verify source data,Poisson GLM
Lakshadweep,2023,59.6,2027,497.9,+734.7%,Large difference; verify source data,Poisson GLM
Lakshadweep,2023,59.6,2026,486.3,+715.3%,Large difference; verify source data,Poisson GLM
Lakshadweep,2023,59.6,2025,475.0,+696.3%,Large difference; verify source data,Poisson GLM
Lakshadweep,2023,59.6,2024,464.0,+677.8%,Large difference; verify source data,Poisson GLM
Arunachal Pradesh,2023,185.8,2027,653.3,+251.5%,Large difference; verify source data,Poisson GLM
Arunachal Pradesh,2023,185.8,2026,638.1,+243.3%,Large difference; verify source data,Poisson GLM


Forecast values are model outputs, not observed measurements. Treat them as exploratory only; future hospital capacity, population, and policy changes are unavailable.
